# Análisis estadístico OE8 — TerraRover-Gen

Este notebook ejecuta el análisis estadístico completo del OE8.

**Instrucciones de uso en Google Colab:**
1. Sube los 13 archivos CSV al área de archivos del notebook (icono de carpeta a la izquierda → botón de subir archivos):
   - HuskyAgent2 y HuskyHeuristic para los 6 terrenos del OE8.
   - Adicionalmente, HuskyAgent2_metricas_BumpyGround_SCI200.csv para el análisis complementario.
2. Ejecuta las celdas de arriba abajo (Shift+Enter).
3. Los resultados se imprimen en la salida y se guardan en `resultados_OE8.json`.

## 1. Instalación de dependencias

Colab ya trae pandas, numpy y scipy. Solo instalamos statsmodels si hace falta.

In [ ]:
!pip install statsmodels --quiet

## 2. Imports y configuración

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests

TERRAINS = ['V3_F3_02', 'BigRock', 'HardTerrain', 'BumpyGround', 'Complete', 'DeepHoles']
CSV_DIR = '.'  # En Colab los CSVs se suben al directorio raíz
COMPLETE_CLEANUP_STEPS = 50  # Umbral de limpieza simétrica en Test Complete

## 3. Funciones de análisis

In [ ]:
def load_pair(terrain):
    """Carga RL y HEU para un terreno con índices alineados por episodio."""
    rl = pd.read_csv(os.path.join(CSV_DIR, f'HuskyAgent2_metricas_{terrain}.csv'), sep=';')
    heu = pd.read_csv(os.path.join(CSV_DIR, f'HuskyHeuristic_metricas_{terrain}.csv'), sep=';')
    rl = rl.set_index('episodio').sort_index()
    heu = heu.set_index('episodio').sort_index()
    common = rl.index.intersection(heu.index)
    return rl.loc[common], heu.loc[common]

def apply_complete_cleanup(rl, heu, steps_threshold=COMPLETE_CLEANUP_STEPS):
    """Limpieza simétrica para Test Complete."""
    rl_bad = (rl['resultado'] == 'FALL') & (rl['pasos'] <= steps_threshold)
    heu_bad = (heu['resultado'] == 'FALL') & (heu['pasos'] <= steps_threshold)
    bad_seeds = rl.index[rl_bad].union(heu.index[heu_bad])
    keep = rl.index.difference(bad_seeds)
    return rl.loc[keep], heu.loc[keep], len(bad_seeds)

def wilson_ci(successes, n, alpha=0.05):
    """IC95% Wilson para una proporción."""
    lo, hi = proportion_confint(successes, n, alpha=alpha, method='wilson')
    return lo * 100, hi * 100

def mcnemar_paired(rl_success, heu_success):
    """McNemar pareado (binomial exacto)."""
    a = ((rl_success) & (heu_success)).sum()
    b = ((~rl_success) & (heu_success)).sum()
    c = ((rl_success) & (~heu_success)).sum()
    d = ((~rl_success) & (~heu_success)).sum()
    result = mcnemar([[a, b], [c, d]], exact=True)
    return int(a), int(b), int(c), int(d), result.pvalue, result.statistic

def diff_props_paired_ci(b, c, n):
    """IC95% aproximado para diferencia pareada de proporciones."""
    diff = (c - b) / n * 100
    if b + c == 0:
        return diff, diff, diff
    se = np.sqrt((b + c - (c - b) ** 2 / n) / n) / n
    lo = (c - b) / n - 1.96 * se
    hi = (c - b) / n + 1.96 * se
    return diff, lo * 100, hi * 100

def wilcoxon_paired(rl_values, heu_values):
    """Wilcoxon pareado con tamaño de efecto r."""
    rl_vals = np.asarray(rl_values)
    heu_vals = np.asarray(heu_values)
    if (rl_vals - heu_vals == 0).all():
        return None
    try:
        _, p = stats.wilcoxon(rl_vals, heu_vals, zero_method='wilcox', alternative='two-sided')
    except ValueError:
        return None
    n = len(rl_vals)
    z = stats.norm.ppf(1 - p / 2)
    r = abs(z) / np.sqrt(n)
    return {'p': float(p), 'r': float(r),
            'median_rl': float(np.median(rl_vals)),
            'median_heu': float(np.median(heu_vals)),
            'n': int(n)}

## 4. Análisis terreno a terreno

In [ ]:
mcnemar_pvalues = []
mcnemar_labels = []
summary = {}

for terrain in TERRAINS:
    print('=' * 90)
    print(f'TERRENO: {terrain}')
    print('=' * 90)
    
    rl, heu = load_pair(terrain)
    n_orig = len(rl)
    
    if terrain == 'Complete':
        rl, heu, n_excluded = apply_complete_cleanup(rl, heu)
        n = len(rl)
        print(f'\nLimpieza simétrica: {n_excluded} excluidos (FALL con pasos<={COMPLETE_CLEANUP_STEPS})')
        print(f'Brutos: {n_orig} → Válidos: {n}\n')
    else:
        n = n_orig
    
    rl_succ = (rl['resultado'] == 'SUCCESS').values
    heu_succ = (heu['resultado'] == 'SUCCESS').values
    n_rl = int(rl_succ.sum())
    n_heu = int(heu_succ.sum())
    
    rl_rate = n_rl / n * 100
    rl_lo, rl_hi = wilson_ci(n_rl, n)
    heu_rate = n_heu / n * 100
    heu_lo, heu_hi = wilson_ci(n_heu, n)
    
    print(f'RL:  {rl_rate:5.1f}%  IC95% [{rl_lo:5.1f}, {rl_hi:5.1f}]  ({n_rl}/{n})')
    print(f'HEU: {heu_rate:5.1f}%  IC95% [{heu_lo:5.1f}, {heu_hi:5.1f}]  ({n_heu}/{n})')
    print(f'\nResultados RL:  {dict(rl["resultado"].value_counts())}')
    print(f'Resultados HEU: {dict(heu["resultado"].value_counts())}')
    
    a, b, c, d, p_mc, _ = mcnemar_paired(rl_succ, heu_succ)
    diff_obs = (n_rl - n_heu) / n * 100
    _, diff_lo, diff_hi = diff_props_paired_ci(b, c, n)
    print(f'\nMcNemar pareado:')
    print(f'  Tabla 2×2: ambos+={a}, HEU+RL-={b}, RL+HEU-={c}, ambos-={d}')
    print(f'  p = {p_mc:.4f}')
    print(f'  Δ pareada RL-HEU: {diff_obs:+.1f}pp  IC95%: [{diff_lo:+.1f}, {diff_hi:+.1f}]')
    
    mcnemar_pvalues.append(p_mc)
    mcnemar_labels.append(terrain)
    
    both_succ = rl_succ & heu_succ
    n_both = int(both_succ.sum())
    print(f'\nWilcoxon pareado (n={n_both} con SUCCESS en ambos):')
    wilcox = {}
    if n_both >= 5:
        for var in ['pasos', 'tiempo_s', 'distancia_final_m', 'energia_total']:
            res = wilcoxon_paired(rl.loc[both_succ, var], heu.loc[both_succ, var])
            if res is None:
                print(f'  {var:24s}: no calculable')
            else:
                d_str = 'RL<HEU' if res['median_rl'] < res['median_heu'] else 'RL>HEU'
                print(f'  {var:24s}: RL={res["median_rl"]:8.2f}  HEU={res["median_heu"]:8.2f}  ({d_str})  p={res["p"]:.4f}  r={res["r"]:.3f}')
            wilcox[var] = res
    else:
        print(f'  (insuficientes pares)')
        for var in ['pasos', 'tiempo_s', 'distancia_final_m', 'energia_total']:
            wilcox[var] = None
    
    summary[terrain] = {
        'n': n, 'n_excluded': n_orig - n,
        'rl_success': n_rl, 'heu_success': n_heu,
        'rl_rate': rl_rate, 'heu_rate': heu_rate,
        'rl_ci': (rl_lo, rl_hi), 'heu_ci': (heu_lo, heu_hi),
        'mcnemar_p': p_mc, 'mcnemar_table': (a, b, c, d),
        'diff_pp': diff_obs, 'diff_ci': (diff_lo, diff_hi),
        'n_both_success': n_both, 'wilcoxon': wilcox,
        'results_rl': dict(rl['resultado'].value_counts()),
        'results_heu': dict(heu['resultado'].value_counts()),
    }
    print()

## 5. Corrección Holm-Bonferroni

In [ ]:
print('=' * 90)
print('CORRECCIÓN HOLM-BONFERRONI (6 comparaciones McNemar)')
print('=' * 90)
reject, p_adj, _, _ = multipletests(mcnemar_pvalues, alpha=0.05, method='holm')
print(f'{"Terreno":15s}  {"p original":>12s}  {"p Holm":>12s}  {"Significativo":>18s}')
for label, p_o, p_h, rej in zip(mcnemar_labels, mcnemar_pvalues, p_adj, reject):
    print(f'{label:15s}  {p_o:>12.4f}  {p_h:>12.4f}  {("SÍ" if rej else "no"):>18s}')
    summary[label]['mcnemar_p_holm'] = float(p_h)
    summary[label]['mcnemar_significant'] = bool(rej)

## 6. Tabla resumen

In [ ]:
print('=' * 100)
print('TABLA RESUMEN')
print('=' * 100 + '\n')
print(f'{"Terreno":15s} {"n":>4s} {"RL %":>6s} {"IC95% RL":>16s} {"HEU %":>6s} {"IC95% HEU":>16s} {"Δpp":>7s} {"p McN":>9s} {"p Holm":>9s}')
print('-' * 100)
for t in TERRAINS:
    d = summary[t]
    print(f'{t:15s} {d["n"]:>4d} {d["rl_rate"]:>5.1f}  [{d["rl_ci"][0]:5.1f},{d["rl_ci"][1]:5.1f}]   '
          f'{d["heu_rate"]:>5.1f}  [{d["heu_ci"][0]:5.1f},{d["heu_ci"][1]:5.1f}]   '
          f'{d["diff_pp"]:>+6.1f}  {d["mcnemar_p"]:>8.4f}  {d["mcnemar_p_holm"]:>8.4f}')

## 7. Análisis complementario: SCI=100 vs SCI=200 en BumpyGround

Compara el mismo agente RL bajo dos configuraciones del sistema anti-atasco para investigar si los STUCK reflejan un timeout estricto o una limitación de la política aprendida.

In [ ]:
print('=' * 90)
print('ANÁLISIS COMPLEMENTARIO: Stuck Check Interval (SCI) en BumpyGround')
print('=' * 90)

bg100 = pd.read_csv(os.path.join(CSV_DIR, 'HuskyAgent2_metricas_BumpyGround.csv'),
                    sep=';').set_index('episodio').sort_index()
bg200 = pd.read_csv(os.path.join(CSV_DIR, 'HuskyAgent2_metricas_BumpyGround_SCI200.csv'),
                    sep=';').set_index('episodio').sort_index()
common = bg100.index.intersection(bg200.index)
bg100 = bg100.loc[common]
bg200 = bg200.loc[common]
n = len(bg100)

s100 = (bg100['resultado'] == 'SUCCESS').values
s200 = (bg200['resultado'] == 'SUCCESS').values
n100 = int(s100.sum())
n200 = int(s200.sum())
rate100 = n100/n*100
rate200 = n200/n*100
lo100, hi100 = wilson_ci(n100, n)
lo200, hi200 = wilson_ci(n200, n)

print(f'SCI=100 (estándar): {rate100:5.1f}%  IC95% [{lo100:5.1f}, {hi100:5.1f}]  ({n100}/{n})')
print(f'SCI=200:            {rate200:5.1f}%  IC95% [{lo200:5.1f}, {hi200:5.1f}]  ({n200}/{n})')

print('\nDistribución de resultados terminales:')
print(f'  {"Resultado":12s} {"SCI=100":>10s} {"SCI=200":>10s} {"Cambio":>10s}')
for r in ['SUCCESS', 'STUCK', 'FALL', 'COLLISION']:
    c1 = (bg100['resultado']==r).sum()
    c2 = (bg200['resultado']==r).sum()
    print(f'  {r:12s} {c1:>10d} {c2:>10d} {c2-c1:>+10d}')

a = int((s100 & s200).sum())
b = int((~s100 & s200).sum())
c = int((s100 & ~s200).sum())
d = int((~s100 & ~s200).sum())
result = mcnemar([[a, b], [c, d]], exact=True)
diff_obs = (n200 - n100)/n*100
_, diff_lo, diff_hi = diff_props_paired_ci(c, b, n)

print(f'\nMcNemar pareado (SCI=100 vs SCI=200):')
print(f'  Tabla 2×2: ambos+={a}, SCI200+/SCI100-={b}, SCI100+/SCI200-={c}, ambos-={d}')
print(f'  p-valor = {result.pvalue:.4f}')
print(f'  Δ SCI200-SCI100: {diff_obs:+.1f}pp  IC95%: [{diff_lo:+.1f}, {diff_hi:+.1f}]')

print('\nInterpretación:')
if result.pvalue < 0.05:
    print(f'  Diferencia estadísticamente significativa (p<0.05).')
else:
    print(f'  Diferencia NO significativa al nivel α=0.05 (p={result.pvalue:.3f}).')
print(f'  Aun con SCI=200, el RL alcanza solo {rate200:.0f}% (vs heurístico 70%),')
print(f'  lo que indica que la limitación está en la política, no en el timeout.')

sci_summary = {
    'n': n,
    'sci100_success': n100, 'sci200_success': n200,
    'sci100_rate': rate100, 'sci200_rate': rate200,
    'sci100_ci': (lo100, hi100), 'sci200_ci': (lo200, hi200),
    'mcnemar_p': float(result.pvalue),
    'mcnemar_table': (a, b, c, d),
    'diff_pp': diff_obs, 'diff_ci': (diff_lo, diff_hi),
    'results_sci100': dict(bg100['resultado'].value_counts()),
    'results_sci200': dict(bg200['resultado'].value_counts()),
}

## 8. Exportación a JSON

In [ ]:
def make_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_serializable(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj

full = {'OE8': summary, 'SCI_variant_BumpyGround': sci_summary}
with open('resultados_OE8.json', 'w', encoding='utf-8') as f:
    json.dump(make_serializable(full), f, indent=2, ensure_ascii=False)
print('Resultados guardados en resultados_OE8.json')